# Gitanjali

## Collecting BN-EN title mapping

In [ ]:
from bs4 import BeautifulSoup
import requests
import json
import re

index_page = requests.get("https://www.milansagar.com/gitanjali/suchiEng.html")
index_page.encoding = 'utf-8' # Crucial for handling the Bengali characters cleanly

soup = BeautifulSoup(index_page.text, 'html.parser')
all_links = soup.find_all('a', href=re.compile(r"(english|bangla)\d"))

# Need this function as some of the entries were spliced across different hyperlinks! What a psycho!
def combine_links(en_link_ids, bn_link_ids):
    en_texts = []
    for idx in en_link_ids:
        en_texts.append(all_links[idx].text)
    combined_en_text = "".join(en_texts)
    
    bn_texts = []
    for idx in bn_link_ids:
        bn_texts.append(all_links[idx].text)
    combined_bn_text = "".join(bn_texts)

    return combined_en_text.strip(), combined_bn_text.strip()

idx = 0
entries = []
while idx < len(all_links):
    en_links = [idx]
    idx += 1
    while (idx < len(all_links)) and ('bangla' not in all_links[idx].attrs['href']):
        en_links.append(idx)
        idx += 1
    
    bn_links = [idx]
    idx += 1
    while (idx < len(all_links)) and ('english' not in all_links[idx].attrs['href']):
        bn_links.append(idx)
        idx += 1

    en_title, bn_title = combine_links(en_links, bn_links)
    entries.append({
        "english_title": en_title,
        "bengali_title": bn_title,
    })

print(f"Successfully extracted {len(entries)} parallel poem mappings ...")
print('Sample Entry...')
print(json.dumps(entries[0], ensure_ascii=False, indent=2))

with open("../data/parsed_source/gitanjali_mapping.json", "w", encoding="utf-8") as f:
    json.dump(entries, f, ensure_ascii=False, indent=2)

## Collecting Bengali poem entries

In [ ]:
import pandas as pd
import unicodedata

poem_texts = pd.read_csv('../data/raw_source/poem.csv', encoding="utf-8")
mappings = pd.read_json('../data/parsed_source/gitanjali_mapping.json', encoding="utf-8")

poem_texts['content'] = poem_texts.content.str.replace('\n', ' ')

poem_texts['browser_safe_text'] = poem_texts['content'].apply(
    lambda x: unicodedata.normalize('NFKC', str(x)).strip() if pd.notnull(x) else ""
)

mappings['browser_safe_text'] = mappings['bengali_title'].apply(
    lambda x: unicodedata.normalize('NFKC', str(x)).strip() if pd.notnull(x) else ""
)
mappings.drop()

full_bn_text = []
for row in mappings.itertuples():
    search_title = row.browser_safe_text
    res_df = poem_texts[poem_texts.browser_safe_text.str.contains(search_title)]
    if res_df.shape[0] == 1:
        full_bn_text.append(res_df.iloc[0].content)
    else:
        # I will collect these manually (there's only 37 of them)
        full_bn_text.append("")

mappings["bengali_version"] = full_bn_text
mappings.drop(columns=['bengali_title'], inplace=True)
mappings.rename(columns={'browser_safe_text': 'bengali_title'}, inplace=True)
mappings.to_csv("../data/parsed_source/bengali_checkpoint.csv", index=False)

# I manually copied the missing ones from https://www.tagoreweb.in/

## Parsing English Translations

In [ ]:
from pathlib import Path
import pandas as pd
import re

bengali_checkpoint = pd.read_csv("../data/parsed_source/bengali_checkpoint.csv")
content = Path('../data/raw_source/gitanjali_en.txt').read_text()

# Parsing english translations
items = re.split(r'\n\s*\d{1,3}\.\s*', content)
items = [item.strip() for item in items if item.strip()]
items = re.findall(r'\d{1,3}\.\s*(.*?)(?=\n\s*\d{1,3}\.|$)', content, re.DOTALL)

english_versions_df = pd.DataFrame(data=items, columns=['english_version'])

mapped_versions = []
for row in bengali_checkpoint.itertuples():
    mapped_versions.append(english_versions_df[english_versions_df.english_version.str.contains(
        row.english_title)].iloc[0].english_version)

bengali_checkpoint['english_version'] = mapped_versions
bengali_checkpoint.to_csv("../data/parsed_source/parallel_text.csv", index=False)

# Shakespeare

In [ ]:
from pathlib import Path
import pandas as pd
import re
content = Path('../data/raw_source/shakespeares-sonnets.txt').read_text()

items = re.split(r'\n\s*\d{1,3}\s*', content)
items = [item.strip() for item in items if item.strip()]
items = re.findall(r'\d{1,3}\s*(.*?)(?=\n\s*\d{1,3}|$)', content, re.DOTALL)

pd.DataFrame(data=items, columns=['sonnet']).to_csv("../data/parsed_source/shakespeare_parsed.csv", index=False)